# functional-module-wrap — faded example 3: Fill the kernel parameter of a Conv2d wrap

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `functional-module-wrap`. The last cell reports your progress on the `PyTorch: functional module wrap` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: functional module wrap` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`functional-module-wrap`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "functional-module-wrap"
DD_SUBTOPIC = "PyTorch: functional module wrap"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A parameterized functional wrap owns its kernel as an `nn.Parameter` so it appears in `parameters()` and receives gradients; `forward` delegates to `F.conv2d`. Registering the weight as a plain tensor (not a Parameter) would silently exclude it from the optimizer.

## Faded exercise 3

Implement `MyConv2d(nn.Module)` (no bias). `__init__` must register the convolution kernel of shape `(out_ch, in_ch, k, k)` as an `nn.Parameter` named `self.weight`; `forward` calls `F.conv2d`. Complete the blanked weight registration.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn
import torch.nn.functional as F

t.manual_seed(5)

class MyConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, k, stride=1, padding=0):
        super().__init__()
        self.stride = stride
        self.padding = padding
        self.weight = None  # TODO: fill in this step — read the prompt cell above

    def forward(self, x):
        return F.conv2d(x, self.weight, stride=self.stride, padding=self.padding)

m = MyConv2d(3, 4, 3, padding=1)
print(m(t.randn(1, 3, 8, 8)).shape)


def _test():
    m = MyConv2d(3, 4, 3, padding=1)
    # the weight must be a registered Parameter of the right shape
    assert isinstance(m.weight, nn.Parameter), type(m.weight)
    assert tuple(m.weight.shape) == (4, 3, 3, 3), m.weight.shape
    # exactly one parameter (the kernel) shows up
    assert sum(1 for _ in m.parameters()) == 1
    # output shape matches a same-config nn.Conv2d, and values match after copying weights
    x = t.randn(1, 3, 8, 8)
    ref = nn.Conv2d(3, 4, 3, padding=1, bias=False)
    with t.no_grad():
        ref.weight.copy_(m.weight)
    out = m(x)
    assert out.shape == ref(x).shape == (1, 4, 8, 8)
    assert t.allclose(out, ref(x), atol=1e-5)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn
import torch.nn.functional as F

t.manual_seed(5)

class MyConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, k, stride=1, padding=0):
        super().__init__()
        self.stride = stride
        self.padding = padding
        self.weight = nn.Parameter(t.randn(out_ch, in_ch, k, k))

    def forward(self, x):
        return F.conv2d(x, self.weight, stride=self.stride, padding=self.padding)

m = MyConv2d(3, 4, 3, padding=1)
print(m(t.randn(1, 3, 8, 8)).shape)
```
</details>